# Chapter 6 Lab: Representations, Embeddings, and Transfer

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import scipy.spatial.distance as distance

%matplotlib inline

# Seed for reproducibility
np.random.seed(42)

## Word Co-occurrence Embeddings

We build a tiny co-occurrence matrix from a corpus of sentences,
apply SVD to create 2D word embeddings, and visualize how similar words cluster.

In [ ]:
# Small corpus
sentences = [
    "king queen castle throne",
    "cat dog animal pet",
    "king golden crown",
    "dog loyal friend pet",
    "queen royal castle kingdom"
]

# Build vocabulary
vocab = set()
for sent in sentences:
    vocab.update(sent.split())
vocab = sorted(list(vocab))
print(f"Vocabulary: {vocab}")
print(f"Vocab size: {len(vocab)}\n")

# Build co-occurrence matrix
# Count how often two words appear within same sentence (context window)
word_to_idx = {w: i for i, w in enumerate(vocab)}
co_occur = np.zeros((len(vocab), len(vocab)))

window_size = 2  # context window
for sent in sentences:
    words = sent.split()
    for i, word in enumerate(words):
        # Define context: words within window_size positions
        start = max(0, i - window_size)
        end = min(len(words), i + window_size + 1)
        for j in range(start, end):
            if i != j:
                wi = word_to_idx[words[i]]
                wj = word_to_idx[words[j]]
                co_occur[wi, wj] += 1

print("Co-occurrence matrix (first 5x5):")
print(co_occur[:5, :5])
print()

# Apply SVD to get embeddings
from numpy.linalg import svd
U, S, Vt = svd(co_occur, full_matrices=False)

# Take first 2 dimensions
embeddings_2d = U[:, :2] * np.sqrt(S[:2])

print(f"Embeddings shape: {embeddings_2d.shape}")
print(f"Word embeddings (first 5):")
for i, w in enumerate(vocab[:5]):
    print(f"  {w:8s}: {embeddings_2d[i]}")
print()

# Plot embeddings
fig, ax = plt.subplots(figsize=(10, 8))

# Color by word category
colors = {}
for w in vocab:
    if w in ["king", "queen", "castle", "crown", "throne", "royal", "kingdom"]:
        colors[w] = "#FF6B6B"  # Royal/monarchy
    elif w in ["cat", "dog", "animal", "pet", "loyal", "friend"]:
        colors[w] = "#4ECDC4"  # Animals/pets
    else:
        colors[w] = "#95E1D3"  # Neutral

for i, w in enumerate(vocab):
    ax.scatter(embeddings_2d[i, 0], embeddings_2d[i, 1], s=300, alpha=0.7, color=colors[w])
    ax.annotate(w, (embeddings_2d[i, 0], embeddings_2d[i, 1]), 
                fontsize=10, ha="center", va="center", fontweight="bold")

ax.set_xlabel("SVD Dimension 1", fontsize=12)
ax.set_ylabel("SVD Dimension 2", fontsize=12)
ax.set_title("Word Embeddings from Co-occurrence Matrix (2D SVD)", fontsize=13)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color="k", linestyle="-", linewidth=0.5)
ax.axvline(x=0, color="k", linestyle="-", linewidth=0.5)

plt.tight_layout()
plt.show()

print("Notice: Related words (royal vs. animals) cluster in different regions.")

## Embedding Geometry: Cosine Similarity

We compute pairwise cosine similarity between word embeddings to show
that semantically similar words have higher similarity scores.

In [ ]:
# Compute cosine similarity
similarities = []
pairs = []

for i, w1 in enumerate(vocab):
    for j, w2 in enumerate(vocab):
        if i < j:
            sim = 1 - distance.cosine(embeddings_2d[i], embeddings_2d[j])
            pairs.append((w1, w2))
            similarities.append(sim)

similarities = np.array(similarities)

# Show specific pairs
specific_pairs = [("king", "queen"), ("cat", "dog"), ("king", "cat"), ("pet", "animal")]

print("Cosine Similarities:")
for p1, p2 in specific_pairs:
    idx1 = word_to_idx[p1]
    idx2 = word_to_idx[p2]
    sim = 1 - distance.cosine(embeddings_2d[idx1], embeddings_2d[idx2])
    print(f"  {p1:8s} -- {p2:8s}: {sim:.4f}")

print()
print(f"Similarity statistics:")
print(f"  Mean:   {similarities.mean():.4f}")
print(f"  Min:    {similarities.min():.4f}")
print(f"  Max:    {similarities.max():.4f}")
print()

# Plot similarity heatmap
fig, ax = plt.subplots(figsize=(10, 9))

# Build full similarity matrix
sim_matrix = np.ones((len(vocab), len(vocab)))
for i in range(len(vocab)):
    for j in range(i + 1, len(vocab)):
        sim = 1 - distance.cosine(embeddings_2d[i], embeddings_2d[j])
        sim_matrix[i, j] = sim
        sim_matrix[j, i] = sim

im = ax.imshow(sim_matrix, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(np.arange(len(vocab)))
ax.set_yticks(np.arange(len(vocab)))
ax.set_xticklabels(vocab, rotation=45, ha="right")
ax.set_yticklabels(vocab)

# Add text annotations
for i in range(len(vocab)):
    for j in range(len(vocab)):
        text = ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center",
                       color="white" if abs(sim_matrix[i, j]) > 0.5 else "black", fontsize=8)

ax.set_title("Cosine Similarity Matrix Between Word Embeddings", fontsize=13)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Cosine Similarity", fontsize=11)
plt.tight_layout()
plt.show()

## Transfer Learning vs Training from Scratch

We train an MLP on digits 0-4 (source task), extract hidden layer features,
and use them as a pretrained representation on digits 5-9 (target task).
We compare with training from scratch on raw pixels.

In [ ]:
# Load digits dataset
digits = load_digits()
X_all, y_all = digits.data, digits.target

# Normalize
scaler = StandardScaler()
X_all_scaled = scaler.fit_transform(X_all)

# Split source (0-4) and target (5-9)
X_source = X_all_scaled[y_all < 5]
y_source = y_all[y_all < 5]

X_target = X_all_scaled[y_all >= 5]
y_target = y_all[y_all >= 5]

print(f"Source task (digits 0-4): {X_source.shape[0]} samples")
print(f"Target task (digits 5-9): {X_target.shape[0]} samples")
print()

# Train source model (MLP on 0-4)
source_model = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation="relu",
    max_iter=1000,
    random_state=42
)
source_model.fit(X_source, y_source)
source_acc = source_model.score(X_source, y_source)
print(f"Source model accuracy on 0-4: {source_acc:.4f}\n")

# Extract hidden layer features from trained source model
def get_hidden_features(model, X):
    """Extract activations from first hidden layer."""
    return model._forward_pass_fast(X)[1]  # Return hidden layer activation

# For sklearn MLPClassifier, use the transform method approach
# We'll manually compute: z = X @ W1 + b1, then a = relu(z)
W1 = source_model.coefs_[0]  # Weights of first layer
b1 = source_model.intercepts_[0]  # Biases of first layer

def relu(z):
    return np.maximum(0, z)

X_source_features = relu(X_source @ W1 + b1)
X_target_features = relu(X_target @ W1 + b1)

print(f"Extracted hidden features shape: {X_source_features.shape}")
print(f"Hidden layer has {X_source_features.shape[1]} units\n")

# Test on target task with limited labeled data (20 samples)
n_labeled = 20
X_target_labeled = X_target_features[:n_labeled]
y_target_labeled = y_target[:n_labeled]

X_target_test_features = X_target_features[n_labeled:]
y_target_test = y_target[n_labeled:]

# Approach 1: Train from scratch on raw pixels (limited data)
X_target_raw = X_target[:n_labeled]  # Raw pixel data
model_scratch = LogisticRegression(max_iter=1000, random_state=42)
model_scratch.fit(X_target_raw, y_target_labeled)
acc_scratch = model_scratch.score(X_target[n_labeled:], y_target_test)

# Approach 2: Use transferred features
model_transfer = LogisticRegression(max_iter=1000, random_state=42)
model_transfer.fit(X_target_labeled, y_target_labeled)
acc_transfer = model_transfer.score(X_target_test_features, y_target_test)

print(f"Results with {n_labeled} labeled target examples:")
print(f"  From scratch (raw pixels): {acc_scratch:.4f}")
print(f"  Transfer learning (hidden features): {acc_transfer:.4f}")
print(f"  Improvement: {acc_transfer - acc_scratch:.4f}")

## Linear Probe: Testing Representation Quality

A linear probe (logistic regression on frozen features) measures representation quality
without the confound of downstream model capacity. We freeze the source features
and train only a linear layer on top.

In [ ]:
# Train various linear probes
accuracies = {}

# Probe 1: On source task (sanity check)
probe_source = LogisticRegression(max_iter=1000, random_state=42)
probe_source.fit(X_source_features, y_source)
acc_source_probe = probe_source.score(X_source_features, y_source)
accuracies["Source task"] = acc_source_probe

# Probe 2: On target task (measure transfer)
probe_target = LogisticRegression(max_iter=1000, random_state=42)
probe_target.fit(X_target_features, y_target)
acc_target_probe = probe_target.score(X_target_features, y_target)
accuracies["Target task (full data)"] = acc_target_probe

# Probe 3: On target task with limited data
probe_target_limited = LogisticRegression(max_iter=1000, random_state=42)
probe_target_limited.fit(X_target_labeled, y_target_labeled)
acc_target_probe_limited = probe_target_limited.score(X_target_test_features, y_target_test)
accuracies[f"Target task ({n_labeled} samples)"] = acc_target_probe_limited

print("Linear Probe Accuracies (Frozen Source Features):")
for task, acc in accuracies.items():
    print(f"  {task:30s}: {acc:.4f}")
print()
print(f"Interpretation: The source MLP learned representations (from digits 0-4)")
print(f"that transfer reasonably well to digits 5-9 via a simple linear layer.")
print(f"This shows the hidden layer captures digit-relevant structure.")

## Representation Quality Varies with Bottleneck

We train autoencoders (PCA with varying bottleneck dimensions: 2, 10, 50)
and examine how reconstruction error and embedding separability change.

In [ ]:
# Use original (unnormalized) digits for cleaner reconstruction error
X_digits = load_digits().data

components_list = [2, 10, 50]
results = {}

for n_comp in components_list:
    pca = PCA(n_components=n_comp, random_state=42)
    X_reduced = pca.fit_transform(X_digits)
    X_reconstructed = pca.inverse_transform(X_reduced)
    
    # Reconstruction error (MSE)
    recon_error = np.mean((X_digits - X_reconstructed) ** 2)
    
    # Explained variance
    explained_var = pca.explained_variance_ratio_.sum()
    
    results[n_comp] = {
        "pca": pca,
        "reduced": X_reduced,
        "reconstructed": X_reconstructed,
        "recon_error": recon_error,
        "explained_var": explained_var
    }

# Print results
print("Autoencoder (PCA) Results:")
for n_comp in components_list:
    r = results[n_comp]
    print(f"  {n_comp:2d} components: "
          f"MSE = {r['recon_error']:.2f}, "
          f"Explained Var = {r['explained_var']:.4f}")

print()

# Plot reconstruction error
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Panel 1: Reconstruction error vs bottleneck size
ax = axes[0, 0]
errors = [results[nc]["recon_error"] for nc in components_list]
ax.plot(components_list, errors, "o-", linewidth=2.5, markersize=8, color="#1f77b4")
ax.set_xlabel("Bottleneck Dimensions", fontsize=11)
ax.set_ylabel("Mean Squared Error", fontsize=11)
ax.set_title("Reconstruction Error vs Bottleneck Size", fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xticks(components_list)

# Panel 2: Explained variance
ax = axes[0, 1]
var_exp = [results[nc]["explained_var"] for nc in components_list]
ax.bar(range(len(components_list)), var_exp, color="#ff7f0e", alpha=0.7, edgecolor="black")
ax.set_xticks(range(len(components_list)))
ax.set_xticklabels([str(nc) for nc in components_list])
ax.set_xlabel("Bottleneck Dimensions", fontsize=11)
ax.set_ylabel("Cumulative Explained Variance", fontsize=11)
ax.set_title("Explained Variance vs Bottleneck Size", fontsize=12)
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3, axis="y")

for i, v in enumerate(var_exp):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)

# Panel 3 & 4: 2D embeddings colored by digit
for idx, n_comp in enumerate(components_list[:2]):
    ax = axes[1, idx]
    X_embed = results[n_comp]["reduced"]
    
    if n_comp == 2:
        # Plot directly for 2D
        scatter = ax.scatter(X_embed[:, 0], X_embed[:, 1], c=y_all, cmap="tab10",
                            s=30, alpha=0.6, edgecolors="k", linewidth=0.5)
        ax.set_xlabel("Component 1", fontsize=11)
        ax.set_ylabel("Component 2", fontsize=11)
    else:
        # Use PCA on the 10-D embedding to 2D for visualization
        pca_viz = PCA(n_components=2, random_state=42)
        X_embed_2d = pca_viz.fit_transform(X_embed)
        scatter = ax.scatter(X_embed_2d[:, 0], X_embed_2d[:, 1], c=y_all, cmap="tab10",
                            s=30, alpha=0.6, edgecolors="k", linewidth=0.5)
        ax.set_xlabel("PCA of Representation (Dim 1)", fontsize=11)
        ax.set_ylabel("PCA of Representation (Dim 2)", fontsize=11)
    
    ax.set_title(f"Embedding (bottleneck={n_comp}, colored by digit)", fontsize=12)
    ax.grid(True, alpha=0.3)
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label("Digit Class", fontsize=10)

plt.tight_layout()
plt.show()

## What to Notice

1. **Co-occurrence Embeddings**: Words that co-occur often are pushed close in embedding space. Similar words (king/queen, cat/dog) cluster together.

2. **Embedding Geometry**: Cosine similarity reveals semantic relationships. High similarity for related concepts, low for unrelated ones. This enables analogy reasoning (king - queen ≈ prince - princess).

3. **Transfer Learning Power**: Features learned on a source task (0-4) transfer to a target task (5-9), even with very limited labeled data (20 samples). Transfer beats training from scratch on pixels.

4. **Linear Probe**: A linear layer on frozen source features achieves decent accuracy. This tests whether the representation is useful, independent of downstream model capacity.

5. **Bottleneck Trade-off**: Smaller bottlenecks (2D) compress more aggressively, losing information (high reconstruction error, poor separability). Larger bottlenecks (50D) preserve more structure but lose compressive benefit. The sweet spot depends on task requirements.